# Dominanz & Zyklen bei Chip-Abräumstrategien (m variabel, exakte Rechnung)

Dieses Notebook erzeugt **alle Strategien** (Chipverteilungen) für gegebene
Trefferwahrscheinlichkeiten `p` und Gesamtchipzahl `total_chips`, vergleicht jede
Strategie mit jeder anderen **exakt** (mit `Fraction`) und wertet die
**Dominanzrelation** sowie **nichttransitive 3-Zyklen** aus.

---

## Eingaben (ganz oben im Code)

```python
p = (Fraction(1,2), Fraction(1,3), Fraction(1,6))  # Trefferwahrscheinlichkeiten
total_chips = 6                                    # Summe der Chips je Strategie
```
---

## Was berechnet wird

1. **Alle Strategien**  
   Alle m-Tupel nichtnegativer Zahlen mit Summe `total_chips`.

2. **Exakte Paarvergleiche**  
   Für jedes Paar (A,B) wird $(P_A, P_B, P_U)$ exakt berechnet.  
   - `beats[A]` = Menge der Strategien, die A **schlägt** (`P_A > P_B`)  
   - `loses[A]` = Gegner, gegen die A **verliert** (`P_B > P_A`)  
   - `ties[A]`  = **Remis** (`P_A = P_B`)

3. **Top-5 Strategien (Ranking)**  
   Sortiert nach **Nettodominanz** `|beats| − |loses|` (Tie-Breaker: `|beats|`).

4. **Dominante Strategien**  
   Eine Strategie ist *dominant*, wenn sie **gegen alle anderen gewinnt**
   (also `len(loses[S]) == 0` und `len(beats[S]) > 0`).

5. **Nichttransitive 3-Zyklen**  
   Tripel $(S,T,W)$ mit  $S \succ T$, $T \succ W$, $W \succ S$.  
   Doppelte (Rotationen) werden über eine **kanonische Darstellung** entfernt.

6. **Beispielrechnung**  
   Für den **ersten** gefundenen 3-Zyklus werden die exakten Wahrscheinlichkeiten
   der drei Paarungen ausgegeben.

---

## Konsolenausgabe (Interpretation)

- **Top-5 Strategien**:  
  `schlägt`, `verliert`, `remis` geben die Out-Degrees, In-Degrees und Remisen im
  Dominanzgraphen an. Höherer Nettowert ⇒ „stärker“ im direkten Vergleich.

- **Dominante Strategie(n)**:  
  Wenn es eine gibt, steht sie hier explizit.

- **Gesamtanzahl 3-Zyklen**:  
  Alle gefundenen, **ohne** Duplikate (Rotationen).  
  Direkt darunter: **Beispielausgabe** (max. die ersten 5).  
  → Anzahl der ausgegebenen Beispiele kannst du mit `cycles[:n]` steuern.

- **Beispielrechnung für einen Zyklus**:  
  Exakte Brüche und auf drei Nachkommastellen gerundete Werte für $P_A, P_B, P_U$ der drei Paarungen im Zyklus.
---

In [2]:
from fractions import Fraction
from functools import lru_cache
from itertools import combinations, permutations

# -------------------------------
# Parameter - Eingabe
# -------------------------------
p = (Fraction(1,2), Fraction(1,3), Fraction(1,6))  # Trefferwahrscheinlichkeiten (m Felder)
total_chips = 5                                     # Summe der Chips je Strategie

# -------------------------------
# Exakte Einzelspiel-Wahrscheinlichkeiten (allgemeines m)
# -------------------------------
def exact_probabilities_fraction(p, A, B):
    p = tuple(Fraction(pi) for pi in p)
    A = tuple(A); B = tuple(B)

    @lru_cache(None)
    def P_A(V, W):
        sum_V, sum_W = sum(V), sum(W)
        if sum_V == 0 and sum_W == 0: return Fraction(0, 1)  # gleichz. Ende -> Remis -> Beitrag 0 für A
        if sum_V == 0: return Fraction(1, 1)                  # A fertig, B nicht -> A gewinnt
        if sum_W == 0: return Fraction(0, 1)                  # B fertig, A nicht -> A verliert

        # "Relevante" Felder (mind. ein Chip liegt irgendwo)
        s = sum(pj for pj, vj, wj in zip(p, V, W) if vj or wj)
        if s == 0: return Fraction(0, 1)

        acc = Fraction(0, 1)
        for j, pj in enumerate(p):
            if V[j] or W[j]:
                Vn = V[:j] + (max(0, V[j]-1),) + V[j+1:]
                Wn = W[:j] + (max(0, W[j]-1),) + W[j+1:]
                acc += (pj / s) * P_A(Vn, Wn)
        return acc

    PA = P_A(A, B)
    PB = P_A(B, A)
    PU = Fraction(1, 1) - PA - PB
    return PA, PB, PU

# -------------------------------
# Alle Strategien (Kompositionen) für m Felder, Summe = total_chips
# -------------------------------
def all_strategies(total, m):
    # rekursiv: alle m-Tupel nichtnegativer Zahlen mit Summe=total
    if m == 1:
        yield (total,)
        return
    for x in range(total + 1):
        for rest in all_strategies(total - x, m - 1):
            yield (x,) + rest

m = len(p)
strategies = list(all_strategies(total_chips, m))

print(f"p = {p}  |  m = {m}  |  total_chips = {total_chips}")
print(f"Anzahl Strategien: {len(strategies)}  (≈ C(m+N-1, m-1))")

# -------------------------------
# Paarvergleiche
# -------------------------------
def compare(A, B):
    PA, PB, PU = exact_probabilities_fraction(p, A, B)
    if PA > PB: return 1, PA, PB, PU  # A dominiert B
    if PB > PA: return -1, PA, PB, PU # B dominiert A
    return 0, PA, PB, PU              # Remis

beats = {S: set() for S in strategies}
loses = {S: set() for S in strategies}
ties  = {S: set() for S in strategies}
probs = {}  # (A,B) -> (PA,PB,PU)

for A, B in combinations(strategies, 2):
    sgn, PA, PB, PU = compare(A, B)
    probs[(A,B)] = (PA, PB, PU)
    probs[(B,A)] = (PB, PA, PU)
    if sgn == 1:
        beats[A].add(B); loses[B].add(A)
    elif sgn == -1:
        beats[B].add(A); loses[A].add(B)
    else:
        ties[A].add(B); ties[B].add(A)

# -------------------------------
# Kennzahlen & Top-5
# -------------------------------
# Nettodominanz-Score: (schlägt - verliert), Tiebreaker: (schlägt)
def score(S):
    return (len(beats[S]) - len(loses[S]), len(beats[S]))

top5 = sorted(strategies, key=score, reverse=True)[:5]
dominant = [S for S in strategies if len(loses[S]) == 0 and len(beats[S]) > 0]

print("\nTop-5 Strategien (nach Nettodominanz, dann 'schlägt'):")
for rank, S in enumerate(top5, 1):
    print(f"{rank:2d}. {S}   | schlägt={len(beats[S])}, verliert={len(loses[S])}, remis={len(ties[S])}")

print("\nDominante Strategie(n):", dominant if dominant else "keine")

# -------------------------------
# Nichttransitive 3-Zyklen (S>T>W>S), bis zu 5 Stück
# Effizientere Suche: gehe Kanten entlang (S->T) und prüfe Kandidaten C in beats[T] mit C->S
# -------------------------------
def canonical_cycle(triple):
    # rotiere, so dass der lexikographisch kleinste Start vorne steht (Richtung fest)
    S,T,W = triple
    rots = [(S,T,W), (T,W,S), (W,S,T)]
    return min(rots)


# ---------- Nichttransitive 3-Zyklen (S>T>W>S), ohne Rotations-Duplikate ----------
cycles_set = set()
for S, T, W in permutations(strategies, 3):
    if T in beats[S] and W in beats[T] and S in beats[W]:
        cycles_set.add(canonical_cycle((S, T, W)))

cycles = sorted(list(cycles_set))

# ---------- Ausgabe ----------
print(f"\nGesamtanzahl 3-Zyklen (S ≻ T ≻ W ≻ S): {len(cycles)}")
print()
print("Beispielausgabe (max. 5):")
for cyc in cycles[:5]:
    print(cyc)

cycles = sorted(list(cycles_set))

# Optional: Beispiel-Wahrscheinlichkeiten für den ersten Zyklus
print()
print("Beispielrechnung für einen Zyklus")
if cycles:
    S,T,W = cycles[0]
    for A,B in [(S,T),(T,W),(W,S)]:
        PA,PB,PU = probs[(A,B)]
        print(f"\n{A} vs {B}:  PA={PA}  PB={PB}  PU={PU}")
        print(f"\n{A} vs {B}:  PA={float(PA):.3f}  PB={float(PB):.3f}  PU={float(PU):.3f} ")
        print()


p = (Fraction(1, 2), Fraction(1, 3), Fraction(1, 6))  |  m = 3  |  total_chips = 5
Anzahl Strategien: 21  (≈ C(m+N-1, m-1))

Top-5 Strategien (nach Nettodominanz, dann 'schlägt'):
 1. (2, 2, 1)   | schlägt=19, verliert=1, remis=0
 2. (3, 1, 1)   | schlägt=19, verliert=1, remis=0
 3. (3, 2, 0)   | schlägt=18, verliert=2, remis=0
 4. (4, 1, 0)   | schlägt=18, verliert=2, remis=0
 5. (2, 3, 0)   | schlägt=16, verliert=4, remis=0

Dominante Strategie(n): keine

Gesamtanzahl 3-Zyklen (S ≻ T ≻ W ≻ S): 6

Beispielausgabe (max. 5):
((0, 3, 2), (0, 4, 1), (3, 0, 2))
((0, 3, 2), (1, 4, 0), (3, 0, 2))
((0, 4, 1), (3, 0, 2), (1, 2, 2))
((1, 2, 2), (1, 4, 0), (3, 0, 2))
((2, 2, 1), (3, 1, 1), (4, 1, 0))

Beispielrechnung für einen Zyklus

(0, 3, 2) vs (0, 4, 1):  PA=131/243  PB=112/243  PU=0

(0, 3, 2) vs (0, 4, 1):  PA=0.539  PB=0.461  PU=0.000 


(0, 4, 1) vs (3, 0, 2):  PA=961/1875  PB=914/1875  PU=0

(0, 4, 1) vs (3, 0, 2):  PA=0.513  PB=0.487  PU=0.000 


(3, 0, 2) vs (0, 3, 2):  PA=114739/337